In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn nltk openpyxl spacy
!python -m spacy download en_core_web_sm

# en la sección "Cargar Dataset", cambia 'data/...' por '/content/...'

In [ ]:
!pip install bertopic

In [ ]:
nltk.download('punkt_tab')

# Middle Term Test 2 - COVID-19 Fake News Detection

## 1.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import FreqDist
from nltk.util import ngrams
from nltk.text import Text
import spacy
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic import BERTopic

## 1.2 Load Dataset

In [ ]:
# cargar datasets test, train, val, y text_with_labels
''' train_df = pd.read_excel('data/Constraint_English_Train.xlsx')
val_df = pd.read_excel('data/Constraint_English_Val.xlsx')
test_df = pd.read_excel('data/Constraint_English_Test.xlsx')
text_with_labels = pd.read_excel('data/english_test_with_labels.xlsx') '''

train_df = pd.read_excel('/content/Constraint_English_Train.xlsx')
val_df = pd.read_excel('/content/Constraint_English_Val.xlsx')
test_df = pd.read_excel('/content/Constraint_English_Test.xlsx')
text_with_labels = pd.read_excel('/content/english_test_with_labels.xlsx')

# info basica
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

# mostrar tweets completos
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

train_df.head(15)

As a simple analysis we can see that there are shorter and longer tweets but most use hashtags and links to sources. We can also see that some use figures to describe what they say, in this small sample all those using figures are real but we will analyze more things below.

## 1.3 EDA (linguistic analysis)

### 1.3.1 Class distribution analysis

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

In [ ]:
df = train_df.copy()
df = df[['tweet', 'label']]

In [ ]:
# distribución de clases
df['label'].value_counts()

In [ ]:
sns.countplot(data=df, x='label')
plt.title("Distribución de clases (real vs fake)")
plt.show()

There are more real than fake tweets, this will give us problems in numerical comparisons since they will be unbalanced.

### 1.3.4 Tokenization + stopwords + frequency

In [ ]:
stop_words = set(stopwords.words("english"))

In [ ]:
def preprocess_nltk(text):
    tokens = word_tokenize(text)
    return [w.lower() for w in tokens if w.isalpha() and w.lower() not in stop_words]

In [ ]:
df['tokens'] = df['tweet'].astype(str).apply(preprocess_nltk)
df[['tweet', 'tokens']].head()

### 1.3.2 Tweet length

In [ ]:
df['tweet_length'] = df['tokens'].apply(len)

In [ ]:
# estadisticas descriptivas por clase
df.groupby('label')['tweet_length'].describe()

The average is 12 for fake tweets and 16 for real ones, not much difference.

The maximum length of fake tweets is larger (835) but let's see why below:

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df, x='label', y='tweet_length')
plt.title("Distribución de la longitud de los tweets")
plt.ylabel("Número de palabras")
plt.xlabel("Tipo de tweet")
plt.show()

Fake tweets have several outliers or anomalies which are tweets with extremely large word counts.

In [ ]:
df[df['tweet_length'] > 100][['tweet', 'label']].head()

Now looking at them more closely they appear to be many tweets separated by \n as if they were not individual tweets but compilations or summaries of hoaxes since they are fake.

### 1.3.3 Lexical diversity

In [ ]:
def lexical_diversity(tokens):
    return len(set(tokens)) / len(tokens) if len(tokens) > 0 else 0

In [ ]:
tokens_real = [w for tokens in df[df['label'] == 'real']['tokens'] for w in tokens]
tokens_fake = [w for tokens in df[df['label'] == 'fake']['tokens'] for w in tokens]
lex_real = lexical_diversity(tokens_real)
lex_fake = lexical_diversity(tokens_fake)

print("Lexical diversity (REAL):", lex_real)
print("Lexical diversity (FAKE):", lex_fake)

Fake tweets have more lexical diversity, probably real tweets repeat each other since real news are the same so they have shared vocabulary.

### 1.3.5 URL cleaning (https)
We need to clean the https tokens because they arise from links.

In [ ]:
def preprocess_nltk_clean(text):
    tokens = word_tokenize(text)
    return [
        w.lower()
        for w in tokens
        if w.isalpha()
        and w.lower() not in stop_words
        and w.lower() != "https"
    ]

In [ ]:
df['tokens_clean'] = df['tweet'].astype(str).apply(preprocess_nltk_clean)

### 1.3.6 Word frequency (by classes)
Here we will identify which words are most related to each class

In [ ]:
tokens_real_clean = [w for tokens in df[df['label'] == 'real']['tokens_clean'] for w in tokens]
tokens_fake_clean = [w for tokens in df[df['label'] == 'fake']['tokens_clean'] for w in tokens]

In [ ]:
fdist_real_clean = FreqDist(tokens_real_clean)
fdist_fake_clean = FreqDist(tokens_fake_clean)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
fdist_real_clean.plot(20, cumulative=False)
plt.title("Top palabras – REAL ")

plt.subplot(1,2,2)
fdist_fake_clean.plot(20, cumulative=False)
plt.title("Top palabras – FAKE ")

plt.tight_layout()
plt.show()

We can see that in fake tweets `coronavirus` predominates as the top word more than 1200 times. And in real tweets `cases` probably are tweets that report new or confirmed cases.

At first glance we can see that other words in fake tweets appear uniformly with no further predominance, meaning they are linguistically rich. While in real tweets we see much more `new`, `tests` or `deaths`.

### 1.3.7 N-grams

In [ ]:
bigrams_real = list(ngrams(tokens_real_clean, 2))
bigrams_fake = list(ngrams(tokens_fake_clean, 2))

trigrams_real = list(ngrams(tokens_real_clean, 3))
trigrams_fake = list(ngrams(tokens_fake_clean, 3))

In [ ]:
fdist_bi_real = FreqDist(bigrams_real)
fdist_bi_fake = FreqDist(bigrams_fake)

In [ ]:
plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
fdist_bi_real.plot(15, cumulative=False)
plt.title("Top bigrams – REAL (sin URLs)")

plt.subplot(1,2,2)
fdist_bi_fake.plot(15, cumulative=False)
plt.title("Top bigrams – FAKE (sin URLs)")

plt.tight_layout()
plt.show()

In n-grams we can see that the most common pair in real tweets is `new, cases` and `confirmed, cases` which tells us that real tweets are more related to new or confirmed cases and for example with `total, number` we can see what we saw at the beginning that they would be more related to putting figures.

As for fake tweets, the second most common is a link and then the word coronavirus probably hashtags that are usually put at the end right after the links like #coronavirus, as third we have `donald, trump` with almost 100 appearances.

We also have `coronavirus, pandemic` in fake tweets which we guess may be hashtags with the aim of drawing attention and alarming.

### 1.3.8 POS tagging

Continuing with linguistic analysis, now we will see analysis by pos tags, meaning word types.
Since there may be more real tweets than fake ones we will not use numbers but we will count and show percentages.

In [ ]:
def pos_distribution(text):
    doc = nlp(text)
    return [token.pos_ for token in doc if token.is_alpha]

In [ ]:
df['pos_tags'] = df['tweet'].astype(str).apply(pos_distribution)

In [ ]:
# por clases
pos_real = [p for tags in df[df['label'] == 'real']['pos_tags'] for p in tags]
pos_fake = [p for tags in df[df['label'] == 'fake']['pos_tags'] for p in tags]
from collections import Counter

# Conteos
pos_real_counts = Counter(pos_real)
pos_fake_counts = Counter(pos_fake)

# Totales
total_pos_real = sum(pos_real_counts.values())
total_pos_fake = sum(pos_fake_counts.values())

# Frecuencias relativas
pos_real_freq = {k: v / total_pos_real for k, v in pos_real_counts.items()}
pos_fake_freq = {k: v / total_pos_fake for k, v in pos_fake_counts.items()}

In [ ]:
pos_df = pd.DataFrame({
    'REAL': pd.Series(pos_real_freq),
    'FAKE': pd.Series(pos_fake_freq)
}).fillna(0)

In [ ]:
pos_df.plot(kind='bar', figsize=(14,6))
plt.title("Distribución relativa de POS (normalizada)")
plt.ylabel("Proporción")
plt.xlabel("Etiqueta POS")
plt.legend()
plt.tight_layout()
plt.show()

We can see that as expected, nouns predominate similarly in both, where we can see the greatest difference is in proper nouns where fake tweets predominate over real ones.

Perhaps this proper noun thing is because there tends to be a more personal language and more emotional or persuasive discourse.

### 1.3.9 Named Entity Recognition (NER)
It's like pos tags but with entities, but we will use nlp to classify tokens into real-world things and we will compare (using percentages).

In [ ]:
def extract_entities(text):
    doc = nlp(text)
    return [ent.label_ for ent in doc.ents]

df['entities'] = df['tweet'].astype(str).apply(extract_entities)

In [ ]:
entities_real = [e for ents in df[df['label'] == 'real']['entities'] for e in ents]
entities_fake = [e for ents in df[df['label'] == 'fake']['entities'] for e in ents]

In [ ]:
ent_real_counts = Counter(entities_real)
ent_fake_counts = Counter(entities_fake)

total_ent_real = sum(ent_real_counts.values())
total_ent_fake = sum(ent_fake_counts.values())

In [ ]:
ent_real_freq = {k: v / total_ent_real for k, v in ent_real_counts.items()}
ent_fake_freq = {k: v / total_ent_fake for k, v in ent_fake_counts.items()}

In [ ]:
ent_df = pd.DataFrame({
    'REAL': pd.Series(ent_real_freq),
    'FAKE': pd.Series(ent_fake_freq)
}).fillna(0)

In [ ]:
ent_df.plot(kind='bar', figsize=(14,6))
plt.title("Distribución relativa de entidades (NER)")
plt.ylabel("Proporción")
plt.xlabel("Tipo de entidad")
plt.legend()
plt.tight_layout()
plt.show()

Here we connect with what we said at the beginning that Real tweets have more numbers or figures than Fake tweets and this can be seen in the difference of the `Cardinal` entity, we also see that real tweets tend to have more dates.

As for Fake tweets they usually have more named persons in their tweets, perhaps it's a mechanism to attract attention as we saw with trump in the n-grams and they also predominate in ORG which are organizations also for the same reason to attract attention surely.

And we can see a very clear predominance in NORP although it is minimal compared to other percentages but they would be like political or religious groups which is logical in fake tweets.

### 1.3.10 Hashtags and mentions

In [ ]:
def count_hashtags(text):
    return sum(1 for w in text.split() if w.startswith("#"))

def count_mentions(text):
    return sum(1 for w in text.split() if w.startswith("@"))

df['num_hashtags'] = df['tweet'].apply(count_hashtags)
df['num_mentions'] = df['tweet'].apply(count_mentions)

In [ ]:
df.groupby('label')[['num_hashtags', 'num_mentions']].mean()

Finally in this analysis, we can see that real tweets have a higher number of hashtags and mentions

## 2 Uninformed Search (BERTopic)
In the previous sections we have performed a linguistic analysis of the dataset to identify global latent patterns in the set of tweets.

But now we will do **topic modelling**, an unsupervised learning technique whose objective is to discover groups of documents that share similar semantic themes without the Real/Fake labels.

### 2.1 Clean the data
We need clean text without the labels (real/fake)

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # eliminar URLs
    text = re.sub(r"[^a-zA-Z\s]", " ", text)        # solo letras
    text = re.sub(r"\s+", " ", text).strip()
    return text.lower()

train_df['cleaned_tweet'] = train_df['tweet'].astype(str).apply(clean_text)

### 2.2 Vector representation: embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

documents = train_df['cleaned_tweet'].tolist()

embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedding_model.encode(documents, show_progress_bar=True)

### 2.3 Elbow method adapted to nlp
Since there is no fixed `k` we will try different n_components in UMAP and measure clustering quality with silhouette score.
Note: clarify that this method is not necessary since it does not have the same effectiveness as with k-means

In [ ]:
sil_scores = []
components_range = [2, 5, 10, 15, 20]

for n in components_range:
    umap_tmp = UMAP(
        n_neighbors=15,
        n_components=n,
        metric='cosine',
        random_state=42
    )
    reduced = umap_tmp.fit_transform(embeddings)

    cluster_tmp = HDBSCAN(min_cluster_size=15)
    labels_tmp = cluster_tmp.fit_predict(reduced)

    # ignoramos ruido (-1)
    mask = labels_tmp != -1
    if len(set(labels_tmp[mask])) > 1:
        score = silhouette_score(reduced[mask], labels_tmp[mask])
        sil_scores.append(score)
    else:
        sil_scores.append(np.nan)

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(components_range, sil_scores, marker='o')
plt.xlabel("Número de dimensiones (UMAP)")
plt.ylabel("Silhouette Score")
plt.title("Selección de dimensionalidad (criterio tipo elbow)")
plt.show()

There is no clear elbow like with k-means but we can get something clear and that is that from 15.0 onwards there is a clear worsening we can say that from a certain number of dimensions adding more information introduces noise and worsens the separability of the clusters.

We can take 10.0 as value because it is in the "plateau" of the curve.

### 2.4 Final dimensionality reduction (UMAP)

In [ ]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    metric='cosine',
    random_state=42
)

### 2.5 HDBSCAN

In [ ]:
cluster_model = HDBSCAN(
    min_cluster_size=15,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

### 2.6 Topics representation

In [ ]:
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)


### 2.6 BERTopic model construction

In [ ]:
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=cluster_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    calculate_probabilities=True,
    language="english",
    verbose=True
)

### 2.7 Unsupervised training

In [ ]:
topics, probs = topic_model.fit_transform(documents)

### 2.8 Initial cluster analysis

In [ ]:
topic_model.get_topic_info()

As we can see it gives us a total of 59 topics plus topic -1 which are: ambiguous tweets, tweets that are too short or tweets that do not clearly fit into any group and simply hdbscan does not force them into any topic.

What we see is:
- Topic: topic name
- Count: number of tweets in that topic
- Representation: most representative words
- Representative_Docs: prototype tweets

Tweets without topic (-1) are 2777 and the largest topics are `0_water_drinking_cure_alcohol, 1_restrictions_uk_boris_england and 2_trump_donald trump_donald_president` with more than 200 tweets.

### 2.9 Topics visualization

In [ ]:
topic_model.visualize_topics()

We can see that there are many differentiated groups, the largest groups are where topic -1 is which are more general things like related to the origin of covid or names given to it (sars, cov..).

The second largest group is more related to politics (trump, belinda..) and misinformation.

There are two medium groups: one is more associated with science or data (records, reported, confirmed..) and the other more general like institutions (schools, hospitals..)

And other small groups like one related to measures (masks, clothes) and another that relates more to countries (italy, brazil..).